# Lab 1 — Grid Operations Agent with Function Tools

**Required · 45 minutes · Level 100**

Build the first version of the internal operations assistant used throughout
this learning path. The agent must look up an incident owner and the first
approved runbook action through deterministic Python tools.

## Learning objectives

- Create an in-process agent with `FoundryChatClient`.
- Describe Python function tools with type annotations.
- Keep operational records and business rules in code, not in the model prompt.
- Verify that the expected tools were actually invoked.

## Prerequisites

Run `az login` (or `az login --use-device-code`) and configure
`FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL`, and
`WORKSHOP_RESOURCE_NAMESPACE` in `.env`.

## Why this pattern?

An agent is useful when the user request is conversational and the model must
decide which tool to call. The tool itself remains deterministic and testable.
Never let a language model perform a real grid switching operation in this lab.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from typing import Annotated

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from pydantic import Field

INCIDENTS = {
    "INC-1042": {
        "asset": "TR-104",
        "owner": "West Operations",
        "severity": "high",
        "runbook": "RB-TRANSFORMER-ALARM",
    }
}
RUNBOOKS = {
    "RB-TRANSFORMER-ALARM": {
        "first_action": "Confirm SCADA alarm state and collect relay evidence.",
        "evidence": "SCADA event ID and relay timestamp",
    }
}
tool_calls: list[tuple[str, str]] = []


def get_incident_owner(
    incident_id: Annotated[
        str, Field(description="Incident identifier, for example INC-1042")
    ],
) -> dict:
    '''Return ownership and asset metadata for a grid incident.'''
    tool_calls.append(("get_incident_owner", incident_id))
    return INCIDENTS.get(incident_id, {"error": "incident not found"})


def get_runbook_first_action(
    runbook_id: Annotated[
        str, Field(description="Approved runbook identifier")
    ],
) -> dict:
    '''Return the first approved diagnostic action and required evidence.'''
    tool_calls.append(("get_runbook_first_action", runbook_id))
    return RUNBOOKS.get(runbook_id, {"error": "runbook not found"})

## Participant task

Edit `PARTICIPANT_GUIDANCE` so the answer is useful to a first-line operator.
Keep these guardrails:

- use tools for operational facts;
- never invent an owner or runbook step;
- never claim that equipment has been operated;
- preserve the four requested output labels.

In [ ]:
# TODO(participant): add one sentence that improves clarity for a first-line operator.
PARTICIPANT_GUIDANCE = "Use short sentences and state what evidence must be recorded."

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)
agent = client.as_agent(
    name=f"grid-ops-guide-{RESOURCE_NAMESPACE}",
    instructions=(
        "You support electricity grid operators. Use both supplied tools for "
        "incident questions. Do not answer operational facts from model memory. "
        "Do not initiate or imply a switching action. If a tool returns an error, "
        "say that the record is unavailable. Format the answer with: INCIDENT, "
        "OWNER, FIRST ACTION, EVIDENCE NEEDED. "
        + PARTICIPANT_GUIDANCE
    ),
    tools=[get_incident_owner, get_runbook_first_action],
)

In [ ]:
question = (
    "For incident INC-1042, identify the responsible team and use its runbook "
    "to give me the first approved diagnostic action and required evidence."
)
result = await agent.run(question)
print(result.text)

## Deterministic success check

In [ ]:
called_tools = {name for name, _ in tool_calls}
assert "get_incident_owner" in called_tools, "The incident lookup tool was not called."
assert "get_runbook_first_action" in called_tools, "The runbook tool was not called."
assert result.text.strip(), "The agent returned no text."
print("PASS — both operational lookups ran and the agent returned an answer.")

## Optional extension

Add an `acknowledge_incident` tool that requires an explicit approval flag.
Keep it in-memory: this workshop must not mutate a production system.

**Expected artifact:** one tool-grounded incident response plus a passing tool-call check.